# Local RAG Chatbot

This notebook explains the implementation of a simple Retrieval-Augmented Generation (RAG) project. Each section contains a short explanation followed by the actual source code.


## Step 1: Create the Vector Database

The `setup_db.py` script prepares the knowledge base for the chatbot.

It performs the following tasks:
- Load PDF, TXT and DOCX files.
- Remove unwanted pages from PDFs.
- Split documents into smaller chunks.
- Generate embeddings using `all-MiniLM-L6-v2`.
- Store embeddings in ChromaDB.


In [ ]:
import os
import glob
import shutil
import torch

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


def build_custom_database():
    print("Initializing Database Setup...")

    db_path = "./local_chroma_db"
    docs_folder = "./my_documents"
    all_documents = []

    if os.path.exists(db_path):
        print("Removing existing vector database...")
        shutil.rmtree(db_path)

    if not os.path.exists(docs_folder):
        os.makedirs(docs_folder)
        print(f"Created folder '{docs_folder}'. Add your documents to this folder and run the script again.")
        return

    print("Scanning documents...")

    files = []
    files.extend(glob.glob(os.path.join(docs_folder, "*.pdf")))
    files.extend(glob.glob(os.path.join(docs_folder, "*.txt")))
    files.extend(glob.glob(os.path.join(docs_folder, "*.docx")))

    if not files:
        print("No supported document files were found.")
        return

    for file in files:
        print(f"Loading: {os.path.basename(file)}")

        if file.endswith(".pdf"):
            loader = PyPDFLoader(file)
            documents = loader.load()

            filtered = []

            for doc in documents:
                text = doc.page_content

                if (
                    "Reading with Insight" in text
                    or "Before you read" in text
                    or "Thinking about the Text" in text
                    or "Exercise" in text
                ):
                    continue

                filtered.append(doc)

            all_documents.extend(filtered)

        elif file.endswith(".txt"):
            all_documents.extend(TextLoader(file, encoding="utf-8").load())

        elif file.endswith(".docx"):
            all_documents.extend(Docx2txtLoader(file).load())

    print(f"Total Documents Loaded: {len(all_documents)}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=60,
        separators=["\n\n", "\n", ".", " "],
    )

    final_chunks = splitter.split_documents(all_documents)

    print(f"Total Chunks Created: {len(final_chunks)}")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("Generating embeddings...")

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device},
    )

    print("Saving Chroma database...")

    db = Chroma.from_documents(
        documents=final_chunks,
        embedding=embeddings,
        persist_directory=db_path,
    )

    db.persist()

    print("Database creation completed successfully.")
    print(f"Database saved at: {os.path.abspath(db_path)}")


if __name__ == "__main__":
    build_custom_database()

## Step 2: Load the Database and Models

The application loads the saved Chroma database, embedding model, cross-encoder and FLAN-T5 model. If the database is missing, the application displays an error message.


## Step 3: Hybrid Retrieval

The application combines two retrieval methods:
- BM25 for keyword matching.
- Vector search for semantic similarity.

The retrieved results are merged using an Ensemble Retriever.


## Step 4: Re-ranking

A Cross Encoder re-ranks the retrieved chunks and keeps only the most relevant documents before sending them to the language model.


## Step 5: Answer Generation

A prompt instructs FLAN-T5 to answer only from the retrieved context. The application also displays the retrieved chunks used to generate the response.


## Step 6: Streamlit Application

The following code implements the complete Streamlit interface.


In [ ]:
import warnings
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

import torch
import streamlit as st
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.chains import RetrievalQA
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from langchain_core.documents import Document
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_core.prompts import PromptTemplate

st.set_page_config(page_title="Local RAG Chatbot")


@st.cache_resource
def load_models_and_db():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    db_path = "./local_chroma_db"

    if not os.path.exists(db_path):
        return None, None, None, None, False

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device},
    )

    vector_store = Chroma(
        persist_directory=db_path,
        embedding_function=embeddings,
    )

    db_data = vector_store.get()

    if not db_data["documents"]:
        return None, None, None, None, False

    saved_chunks = [
        Document(page_content=txt, metadata=m)
        for txt, m in zip(db_data["documents"], db_data["metadatas"])
    ]

    bm25_retriever = BM25Retriever.from_documents(saved_chunks)

    cross_encoder = HuggingFaceCrossEncoder(
        model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        model_kwargs={"device": device},
    )

    model_id = "google/flan-t5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

    pipe = pipeline(
        "text2text-generation",
        model=model,
        tokenizer=tokenizer,
        max_length=256,
        device=0 if device == "cuda" else -1,
    )

    llm = HuggingFacePipeline(pipeline=pipe)

    return vector_store, bm25_retriever, cross_encoder, llm, True


st.title("Document Question Answering System")

vector_store, bm25_retriever, cross_encoder, llm, is_ready = load_models_and_db()

if not is_ready:
    st.error(
        "Database not found or it is empty. Add your files to the 'my_documents' folder and run 'python setup_db.py' first."
    )

else:
    st.sidebar.header("Search Settings")

    top_k_retrieve = st.sidebar.slider(
        "Chunks to Retrieve",
        1,
        10,
        5,
    )

    top_k_rerank = st.sidebar.slider(
        "Chunks to Rerank",
        1,
        5,
        3,
    )

    user_query = st.text_input(
        "Ask a question based on your documents:"
    )

    if st.button("Generate Answer") and user_query:

        with st.spinner("Generating Answer..."):

            vector_retriever = vector_store.as_retriever(
                search_type="mmr",
                search_kwargs={
                    "k": 3,
                    "fetch_k": 8,
                    "lambda_mult": 0.7,
                },
            )

            bm25_retriever.k = 3

            hybrid_retriever = EnsembleRetriever(
                retrievers=[bm25_retriever, vector_retriever],
                weights=[0.3, 0.7],
            )

            reranker = CrossEncoderReranker(
                model=cross_encoder,
                top_n=2,
            )

            final_retriever = ContextualCompressionRetriever(
                base_compressor=reranker,
                base_retriever=hybrid_retriever,
            )

            prompt = PromptTemplate(
                template="""
                You are an AI assistant.
                Use ONLY the retrieved context.
                Do not copy the context verbatim.
                Answer the question in your own words using only the retrieved context.
                If the answer cannot be found, reply exactly:
                I could not find the answer in the provided document.
                Answer in 3-4 complete sentences.

                Context:
                {context}

                Question:
                {question}

                Answer:
                """,
                input_variables=["context", "question"],
            )

            rag_chain = RetrievalQA.from_chain_type(
                llm=llm,
                chain_type="stuff",
                retriever=final_retriever,
                return_source_documents=True,
                chain_type_kwargs={
                    "prompt": prompt,
                },
            )

            response = rag_chain.invoke({"query": user_query})

            print("\n" + "=" * 100)
            print("QUERY:", user_query)
            print("=" * 100)

            for i, doc in enumerate(response["source_documents"]):
                print(f"\nChunk {i + 1}")
                print("-" * 100)
                print(doc.page_content)
                print("-" * 100)

            st.subheader("Answer")
            st.write(response["result"])

            st.subheader("Retrieved Context")

            for i, doc in enumerate(response["source_documents"]):

                st.markdown(f"### Chunk {i + 1}")

                st.code(doc.page_content)

                st.write("---")

## Workflow

```text
Documents
    ↓
Document Loading
    ↓
Filtering
    ↓
Chunking
    ↓
Embedding Generation
    ↓
ChromaDB
    ↓
Hybrid Retrieval
    ↓
Cross Encoder Re-ranking
    ↓
FLAN-T5
    ↓
Final Answer
```
